# 04 - Statistical validation

Segment boundaries drawn by an unsupervised method are a hypothesis. This
stage tests whether the segments actually differ in ion intensity, and by
how much.

> This notebook is a thin wrapper around the `src/` package. Every computation
> below is the same function the CLI calls, so the notebook and
> `python scripts/run_pipeline.py` produce identical results. To change a
> parameter, edit `configs/pipeline_config.yaml` rather than the code here.

In [1]:
import sys
from pathlib import Path

# Locate the project root so `src` imports work wherever Jupyter was started from.
ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
from src.config import load_config

config = load_config()
print("Raw data :", config.raw_dir)
print("Outputs  :", config.processed_dir)

Raw data : C:\Users\amanb\OneDrive\Desktop\dissertation\Fossil Fly\data\raw
Outputs  : C:\Users\amanb\OneDrive\Desktop\dissertation\Fossil Fly\data\processed


In [2]:
import numpy as np

from src.features.selection import build_cube, select_analysis_channels
from src.preprocessing.stack_io import load_clean_stack
from src.segmentation import cluster
from src.viz.style import segment_names

segments = cluster.load_segments(config.processed_dir)
fg_idx = np.where(segments["foreground_mask"].astype(bool))[0]
labels = segments["labels_dominant"]

images, metadata = load_clean_stack(config.processed_dir)
selection = select_analysis_channels(images, metadata, config)
cube = build_cube(images, selection)
cube_fg = cube.reshape(-1, cube.shape[-1])[fg_idx]

names = segment_names(len(set(labels)), config)
print(f"{cube_fg.shape[0]:,} pixels, {cube_fg.shape[1]} channels, "
      f"{len(set(labels))} segments")

48,294 pixels, 5 channels, 3 segments


## Segment chemistry

In [3]:
from src.stats import validation

profiles = validation.segment_chemistry_profiles(
    cube_fg, labels, selection.labels, names
)
profiles.round(2)

,m/z 62.96,m/z 78.95,m/z 96.94,m/z 103.91,m/z 123.95
Segment 1 (EM1),1.69,27.56,19.01,22.80,2284.32
Segment 2 (EM2),669.62,1626.07,64.89,18.68,119.72
Segment 3 (EM3),2.99,92.25,311.28,1375.25,562.73


## Kruskal-Wallis per channel

Non-parametric, because ion-count distributions are heavily zero-inflated.
With tens of thousands of pixels almost anything reaches significance, so the
effect size is what carries the meaning.

In [4]:
results, posthoc = validation.kruskal_dunn_by_channel(
    cube_fg, labels, selection.labels,
    p_adjust=config.get("stats.p_adjust", "bonferroni"),
)
results

,channel,mass_label,H_statistic,p_value,p_display,eta_squared,significant,effect_size,n_pixels,n_segments
0,m/z 62.96 (Neg),m/z 62.96,8855.09,0.0,<1e-50,0.1834,***,Large,48294,3
1,m/z 78.95 (Neg),m/z 78.95,35308.19,0.0,<1e-50,0.7311,***,Large,48294,3
2,m/z 96.94 (Neg),m/z 96.94,6228.32,0.0,<1e-50,0.1290,***,Medium,48294,3
3,m/z 103.91 (Neg),m/z 103.91,27580.78,0.0,<1e-50,0.5711,***,Large,48294,3
4,m/z 123.95 (Neg),m/z 123.95,25030.64,0.0,<1e-50,0.5183,***,Large,48294,3


## Which pairs of segments differ?

Dunn's post-hoc on the channel with the largest effect size.

In [5]:
from src.viz import stats_plots

best = validation.strongest_effect_channel(results)
print(f"largest effect: {best}")

stats_plots.plot_dunn_heatmap(
    posthoc[best], best, config.figure_path("04_dunn_posthoc_heatmap.png")
)
posthoc[best].round(4)

largest effect: m/z 78.95 (Neg)


,0,1,2
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0


## Cross-polarity colocalization

Does the secondary-mode ion concentrate where any segment sits? The secondary
acquisition is resampled onto the analysis grid first.

In [6]:
from src.features.selection import resample_to

coloc = None
if selection.secondary_keys:
    secondary = resample_to(images[selection.secondary_keys[0]], selection.shape)
    secondary_flat = secondary.reshape(-1)[fg_idx]
    coloc = validation.colocalization(secondary_flat, labels, segment_names=names)
    stats_plots.plot_colocalization(
        coloc, config.figure_path("04_cross_polarity_coloc.png")
    )
coloc

,segment,segment_name,pearson_r,p_value,p_display,interpretation
0,0,Segment 1 (EM1),-0.0011,0.805798,0.806,Weak / no colocalization
1,1,Segment 2 (EM2),-0.0069,0.131401,0.131,Weak / no colocalization
2,2,Segment 3 (EM3),0.0090,0.048059,0.048,Weak / no colocalization


In [7]:
written = validation.save_statistics(config.processed_dir, results, posthoc, coloc)
for name, path in written.items():
    print(f"{name:16s} -> {path.name}")

results          -> 04_statistical_results.csv
posthoc          -> 04_dunn_posthoc.csv
colocalization   -> 04_colocalization.csv


Equivalent CLI command:

```bash
python scripts/run_pipeline.py --stage stats
```